## Embedding Model

In [5]:
import torch
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

### Metal Device Init

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    x = torch.ones(1, device=device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


### HuggingFace Init

In [13]:
embed_model_id = 'sentence-transformers/all-MiniLM-L6-v2'

embed_model = HuggingFaceEmbeddings(
    model_name=embed_model_id,
    model_kwargs={'device': device},
    encode_kwargs={'device': device, 'batch_size': 32}
)

### Embedding Generation

In [15]:
ismir_abstracts = [
    "Motivated by musicological applications of the four-way categorization of tabla strokes, we consider automatic classification methods that are potentially robust to instrument differences. We present a new, diverse tabla dataset suitably annotated for the task. The acoustic correspondence between the tabla stroke categories and the common popular Western drum types motivates us to adapt models and methods from automatic drum transcription. We start by exploring the use of transfer learning on a state-of-the-art pre-trained multiclass CNN drums model. This is compared with 1-way models trained separately for each tabla stroke class. We find that the 1-way models provide the best mean f-score while the drums pre-trained and tabla-adapted 3-way models generalize better for the most scarce target class. To improve model robustness further, we investigate both drums and tabla-specific data augmentation strategies.",
    "Some generative models for sequences such as music and text allow us to edit only subsequences, given surrounding context sequences, which plays an important part in steering generation interactively. However, editing subsequences mainly involves randomly resampling subsequences from a possible generation space. We propose a contextual latent space model (CLSM) in order for users to be able to explore subsequence generation with a sense of direction in the generation space, e.g., interpolation, as well as exploring variations—semantically similar possible subsequences. A context-informed prior and decoder constitute the generative model of CLSM, and a context position-informed encoder is the inference model. In experiments, we use a monophonic symbolic music dataset, demonstrating that our contextual latent space is smoother in interpolation than baselines, and the quality of generated samples is superior to baseline models. The generation examples are available online.",
    "Most of the musical heritage is only available as physical documents, given that the engraving process was carried out by handwriting or typesetting until the end of the 20th century. Their mere availability as scanned images does not enable tasks such as indexing or editing unless they are transcribed into a structured digital format. Given the cost and time required for manual transcription, Optical Music Recognition (OMR) presents itself as a promising alternative. Quite often, OMR systems show acceptable but not perfect performance, which eventually leaves them out of the transcription process. On the assumption that OMR systems might always make some errors, it is essential that the user corrects the output. This paper contributes to a better understanding of how music transcription is improved by the assistance of OMR systems that include the end-user in the recognition process. For that, we have measured the transcription time of a printed early music work under two scenarios: a manual one and a state-of-the-art OMR-assisted one, with several alternatives each. Our results demonstrate that using OMR remarkably reduces users' effort, even when its performance is far optimal, compared to the fully manual option."
]

embeddings = embed_model.embed_documents(ismir_abstracts)

for e in embeddings:
    print(f'Length: {len(e)}; {e}')

Length: 384; [0.012115173041820526, -0.13450594246387482, 0.04874710738658905, -0.02112574875354767, -0.08244277536869049, 0.06822236627340317, -0.031022600829601288, -0.13320374488830566, -0.021343054249882698, -0.059392668306827545, -0.03994246572256088, -0.03230874985456467, -0.040683623403310776, -0.016141479834914207, -0.045929793268442154, -0.06799328327178955, 0.05937036871910095, 0.048257410526275635, -0.09676941484212875, -0.039426982402801514, -0.01912565715610981, 0.05380105599761009, 0.062178436666727066, -0.024705959483981133, 0.03187497332692146, 0.024786224588751793, -0.06025942414999008, 0.0021640120539814234, 0.029679978266358376, -0.0971885398030281, -0.08553329855203629, 0.07490863651037216, -0.06415635347366333, -0.027083776891231537, -0.053190916776657104, -0.006843732204288244, -0.01666606403887272, 0.096835196018219, -0.015074243769049644, 0.023609133437275887, 0.03311862796545029, 0.0005762227810919285, 0.017803601920604706, -0.04751021787524223, -0.081691347062

## Pinecone

In [67]:
import os
import pinecone
import time
from dotenv import load_dotenv
import pandas as pd
import uuid
from tqdm.auto import tqdm

### Create pinecone index

In [37]:
load_dotenv()

pinecone.init(
    api_key=os.environ.get("PINECONE_API_KEY"),
    environment=os.getenv("PINECONE_ENVIRONMENT")
)

In [39]:
index_name = 'llama2-ismir2021'

if index_name not in pinecone.list_indexes():
    pinecone.create_index(
        index_name,
        dimension=len(embeddings[0]),
        metric='cosine' # dependent on model
    )
    while not pinecone.describe_index(index_name).status['ready']:
        time.sleep(1)

In [63]:
index = pinecone.Index(index_name)
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

### Embed ismir data

In [59]:
with open("./data/ismir2021.json", "r") as f:
    ismir_df = pd.read_json(f)

ismir_df['uuid'] = ismir_df.apply(lambda _: uuid.uuid4(), axis=1)
ismir_df['combined'] = ismir_df.apply(lambda x: f'title: {x["title"]}, abstract: {x["abstract"]}', axis=1)

ismir_df.head()

,title,author,year,pages,abstract,ee,extra,uuid,combined
0,Four-way Classification of Tabla Strokes with ...,"[Rohit M A, Amitrajit Bhattacharjee, Preeti Rao]",2021,19-26,Motivated by musicological applications of the...,https://archives.ismir.net/ismir2021/paper/000...,{'takeaway': 'Automatic transcription for data...,01e12e14-d85d-4c89-9836-8f33264723b8,title: Four-way Classification of Tabla Stroke...
1,A Contextual Latent Space Model: Subsequence M...,[Taketo Akama],2021,27-34,Some generative models for sequences such as m...,https://archives.ismir.net/ismir2021/paper/000...,{'takeaway': 'Latent space models and context ...,58e5164d-b6aa-4369-a495-0c70a5bb6d6e,title: A Contextual Latent Space Model: Subseq...
2,OMR-assisted transcription: a case study with ...,"[María Alfaro-Contreras, David Rizo, Jose M. I...",2021,35-41,Most of the musical heritage is only available...,https://archives.ismir.net/ismir2021/paper/000...,{'takeaway': 'This paper contributes to a bett...,1c3d7f5d-0bc6-4333-8fce-8cfe8473c3cf,title: OMR-assisted transcription: a case stud...
3,Deeper Convolutional Neural Networks and Broad...,[Stefan A Baumann],2021,42-49,"In recent years, complex convolutional neural ...",https://archives.ismir.net/ismir2021/paper/000...,"{'takeaway': 'Deeper, more complex Convolution...",9b8eff72-c745-4278-822f-a0af9ef53d25,title: Deeper Convolutional Neural Networks an...
4,The Music Performance Markup Format and Ecosystem,[Axel Berndt],2021,50-57,Music Performance Markup (MPM) is a new XML fo...,https://archives.ismir.net/ismir2021/paper/000...,{'takeaway': 'The paper introduces the Music P...,fd7a66ba-1a88-46d0-8d14-12959c63b644,title: The Music Performance Markup Format and...


### Upsert to Pinecone

In [68]:
batch_size = 32

for i in tqdm(range(0, len(ismir_df), batch_size)):
    i_end = min(len(ismir_df), i+batch_size)
    batch = ismir_df.iloc[i:i_end]
    ids = [f"{x['uuid']}" for i, x in batch.iterrows()]
    texts = [x['combined'] for i, x in batch.iterrows()]
    embeds = embed_model.embed_documents(texts)

    metadata = [
        {
            'title': x['title'],
            'authors': x['author'],
            'year': x['year'],
            'abstract': x['abstract'],
            'ee': x['ee'],
            'takeaway': x['extra']['takeaway'],
            'best_paper_candidate_bool': x['extra']['best_paper_candidate'],
            'subject_area_primary': x['extra']['subject_area_primary']
        } for i, x in batch.iterrows()
    ]

    index.upsert(vectors=zip(ids, embeds, metadata))

100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


In [69]:
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.00104,
 'namespaces': {'': {'vector_count': 104}},
 'total_vector_count': 104}

## Hugging Face Pipeline